# Qwen 3.5 4B
Another vision-language model. This time without `pipeline`: we use the processor and model directly.

## Install
**Restart the kernel after installing!**

In [1]:
%pip install -U transformers accelerate bitsandbytes

Looking in indexes: https://nexus.iisys.de/repository/ki-awz-pypi-group/simple, https://pypi.org/simple
Note: you may need to restart the kernel to use updated packages.


## Load the model
Again in 4-bit. The processor turns image + text into model input.

In [2]:
from transformers import AutoModelForMultimodalLM, AutoProcessor, BitsAndBytesConfig
import torch

model = AutoModelForMultimodalLM.from_pretrained(
    "Qwen/Qwen3.5-4B",
    dtype=torch.bfloat16,
    device_map="cuda",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True),
)

processor = AutoProcessor.from_pretrained("Qwen/Qwen3.5-4B")

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

## Helper function
`enable_thinking=False` skips the model's long "thinking" and gives a direct answer.

In [3]:
def generate_answer(prompt, image):
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    # Image + text -> model input
    inputs = processor.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
        return_tensors="pt",
        enable_thinking=False,
    ).to("cuda")

    # Generate, then decode only the new tokens (the answer)
    generated_ids = model.generate(**inputs, max_new_tokens=256)
    answer_ids = generated_ids[0][len(inputs.input_ids[0]):]
    return processor.decode(answer_ids, skip_special_tokens=True)

## Ask
The image can be a URL...

In [4]:
print(generate_answer("Describe this image. Tell me a fun fact about this puppy.", "https://cf.ltkcdn.net/dogs/puppies/images/std/321597-800x515-cutest-puppy-videos.jpg"))

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


This adorable image features a small, fluffy puppy — likely a **Chihuahua** or **Chihuahua mix** — snugly nestled inside a white ceramic bowl decorated with bright red polka dots. The puppy has a black-and-tan coat, large expressive eyes, and its tiny pink paws are peeking out over the rim of the bowl, giving it a comical “cup-holding” appearance. Its tongue is slightly sticking out, adding to the cuteness.

The bowl sits on a patch of artificial green grass, and behind it, there’s a white picket fence and a small white dish with green paint or marker on it — suggesting this might be a staged photo for a pet portrait or social media post.

---

🐾 **Fun Fact About This Puppy:**

If this puppy is indeed a Chihuahua (which it very likely is, given its size and facial structure), then here’s a fun fact:  
> **Chihuahuas are one of the smallest dog breeds in the world**, with some weighing as little as 2–4 pounds! Despite their tiny size, they’re known for being surprisingly bold, confident

...or a local file.

In [5]:
print(generate_answer("What animal is this?", "images/Hase.png"))

The animal in the image is a **rabbit** — specifically, an anthropomorphized rabbit dressed in formal attire (suit, tie, and fedora), standing beside a woman in a vintage-style dress.

This is likely a stylized or surreal image, possibly generated by AI, given the smooth textures, perfect lighting, and the unusual combination of a human and a fully dressed animal character. The setting — a garden with flowers, a wooden house, and a wooden fence — evokes a nostalgic, perhaps 1920s–1930s, aesthetic.

So, while the “animal” is clearly a **rabbit**, it’s presented here as a charming, human-like character — not just a wild animal, but a figure in a story or scene.

🐰 *Bonus: It’s wearing a hat and tie — very “gentleman rabbit”!*

Let me know if you’d like to explore the symbolism, style, or origin of such images!

